# 04 — Building a Text-to-Text Generation System with Transformers

## 📚 Learning Objectives

By completing this notebook, you will:
- Build an **encoder–decoder Transformer** (the T5/BART architecture family) in pure PyTorch
- Understand the seq2seq machinery: SOS/EOS tokens, teacher forcing, the causal target mask, greedy decoding
- Learn why Transformers need **positional encodings** — attention alone is order-blind
- **Evaluate** the system with exact-match and per-character accuracy on held-out *real* sentences, and read the errors it actually makes

## 🔗 Where this fits

**Builds on:** Course 07 (AIAT 121) — Unit 4, lessons 01 and 04 (attention, and seq2seq with an encoder-decoder) — that architecture, now built and trained in PyTorch by the usual recipe: define a loss, then use gradient descent to nudge the parameters.

**Used later in:** Course 10 — Unit 3, lesson 04, where the same conditioning idea plugs text into an image generator.

## 📊 Data used in this notebook

**Real published prose** from the NLTK Gutenberg corpus — six books (Carroll, Austen ×2, Chesterton ×2, Bryant). Every training pair is built from a real sentence:

| side | content |
|---|---|
| **source** | the sentence stripped to lowercase letters and spaces — `so goodby my dear` |
| **target** | the sentence exactly as the author wrote it — `So goodby, my dear.` |

Nothing here is invented: the target is the author's real text, and the source is that same real text with information deliberately removed.

---

## Introduction

**Text-to-text** systems (T5, BART, translation models) frame every task as "text in → text out": an **encoder** reads the input sequence, a **decoder** generates the output autoregressively while attending to the encoder's states.

Our task is **truecasing and punctuation restoration** — turning `i ought to travel` back into `I ought to travel.` This is not a puzzle invented for a notebook: it is a standard production stage in **speech-recognition pipelines**, which emit an unpunctuated lowercase token stream that has to be made readable before a human sees it. Subtitles, voice notes, and meeting transcripts all run a model like this one.

Because the data is real, the task is genuinely **partly ambiguous** — nothing in the stripped string `it will be giving him so much pleasure` tells you that Austen ended it with `!` rather than `.`. Expect held-out exact-match accuracy well below 100% (we measure **49.0%**, against **81.6%** per-character), and read the failures: they are where the lesson is.

**One detail is load-bearing: positional encodings.** Self-attention is *permutation-invariant* — without position information the model cannot represent "first character" vs "last character", so it could not learn to capitalise the *opening* letter or place a mark at the *end*. Our model adds a learned positional embedding to every token; try removing the `add_pos` calls and watch accuracy collapse.

---

## 🌍 Why this lesson exists — a production stage that got deleted

Truecasing and punctuation restoration — the task you are about to build — was a standard, separately-trained stage in every speech-recognition pipeline for years: the recogniser emitted an unbroken lowercase token stream, and this component made it readable before a human saw it. Then OpenAI's **Whisper** (Radford et al., 2022) shipped a model that emits cased, punctuated text directly, and for most pipelines the separate restoration model simply stopped being built.

Both halves of that are worth carrying: the task is genuinely a production stage (subtitles, voice notes, meeting transcripts all depend on it), *and* a well-defined component can be absorbed by a larger end-to-end model in a single release. Knowing which of your components is absorbable is a career skill.

**What goes wrong without this:** without a restoration stage, an ASR system hands a human an unreadable lowercase wall. The task exists only because the information was stripped upstream — which is also why parts of it are unrecoverable, as the results below show.


In [1]:
# WHAT/WHY: build and train a small encoder-decoder Transformer that restores
# capitalisation and punctuation to REAL sentences from published books, then
# measure exact-match and per-character accuracy on held-out sentences.
# Note the positional embeddings — without them, attention cannot see token
# order, and "capitalise the first letter / punctuate the last" is unlearnable.
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import numpy as np
import re
print(f'PyTorch {torch.__version__}')

torch.manual_seed(0); np.random.seed(0)

# ── REAL DATA: six books from the NLTK Gutenberg corpus ───────────────────
# Fallback keeps the notebook runnable offline; both branches are real text.
def load_books():
    try:
        import nltk
        nltk.download('gutenberg', quiet=True)
        from nltk.corpus import gutenberg
        books = ['carroll-alice.txt', 'austen-emma.txt', 'chesterton-brown.txt',
                 'bryant-stories.txt', 'austen-sense.txt', 'chesterton-thursday.txt']
        return [(b, gutenberg.raw(b)) for b in books]
    except Exception:
        from sklearn.datasets import fetch_20newsgroups
        news = fetch_20newsgroups(subset='train', categories=['rec.autos'],
                                  remove=('headers', 'footers', 'quotes'))
        return [('20newsgroups rec.autos (fallback)', " ".join(news.data))]

# ── Build (stripped sentence → original sentence) pairs ───────────────────
# The TARGET is the author's real sentence. The SOURCE is that same sentence
# with case and punctuation removed - exactly what a speech recogniser emits.
pairs = []
for name, raw in load_books():
    raw = re.sub(r'\s+', ' ', raw)
    for s in re.split(r'(?<=[.!?]) ', raw):
        s = s.strip()
        if not (12 <= len(s) <= 40):                       # keep sentences short
            continue
        if not re.fullmatch(r"[A-Za-z ,.'!?]+", s):        # plain prose only
            continue
        src = re.sub(r'\s+', ' ', re.sub(r"[^a-z ]", '', s.lower())).strip()
        if len(src) >= 8:
            pairs.append((src, s))

print(f'Real sentence pairs extracted: {len(pairs):,}')
print('Examples (source → target):')
for a, b in pairs[:4]:
    print(f'  {a!r} → {b!r}')

# ── Character vocabulary over both sides + 3 special tokens ───────────────
chars = sorted(set(''.join(a for a, _ in pairs) + ''.join(b for _, b in pairs)))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
PAD, SOS, EOS = len(chars), len(chars) + 1, len(chars) + 2
V = len(chars) + 3
MAX_SRC = max(len(a) for a, _ in pairs)
MAX_TGT = max(len(b) for _, b in pairs)
print(f'\nVocabulary: {V} tokens ({len(chars)} characters + PAD/SOS/EOS)')
print(f'Longest source: {MAX_SRC} chars | longest target: {MAX_TGT} chars')

# ── Held-out split: 90% train / 10% test, shuffled with a fixed seed ──────
rng = np.random.RandomState(0)
pairs = [pairs[i] for i in rng.permutation(len(pairs))]
split = int(0.9 * len(pairs))
train_pairs, test_pairs = pairs[:split], pairs[split:]
print(f'Train: {len(train_pairs):,} sentences | held-out test: {len(test_pairs):,}')

def encode_src(s):
    return [c2i[c] for c in s] + [PAD] * (MAX_SRC - len(s))

def encode_tgt(s):
    # teacher forcing: decoder INPUT starts with SOS, decoder TARGET ends with EOS
    ids = [c2i[c] for c in s]
    return ([SOS] + ids + [PAD] * (MAX_TGT - len(ids)),
            ids + [EOS] + [PAD] * (MAX_TGT - len(ids)))

X_src   = torch.tensor([encode_src(a) for a, _ in train_pairs], dtype=torch.long)
tgt_in  = torch.tensor([encode_tgt(b)[0] for _, b in train_pairs], dtype=torch.long)
tgt_out = torch.tensor([encode_tgt(b)[1] for _, b in train_pairs], dtype=torch.long)
print(f'Tensors — src {tuple(X_src.shape)} | tgt_in {tuple(tgt_in.shape)}')

# ── Encoder-decoder Transformer with LEARNED positional embeddings ────────
class Seq2SeqTransformer(nn.Module):
    def __init__(self, V, d=64, nhead=4, nlayers=2, max_len=64):
        super().__init__()
        self.src_emb = nn.Embedding(V, d)
        self.tgt_emb = nn.Embedding(V, d)
        self.pos_emb = nn.Embedding(max_len, d)   # position i → learned vector
        self.transformer = nn.Transformer(d, nhead, nlayers, nlayers,
                                          dim_feedforward=128, batch_first=True)
        self.fc = nn.Linear(d, V)
    def add_pos(self, emb):
        # add position information — attention alone is order-blind
        pos = torch.arange(emb.size(1), device=emb.device)
        return emb + self.pos_emb(pos)[None, :, :]
    def forward(self, src, tgt):
        # causal mask: decoder position t may only attend to positions ≤ t
        mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1))
        out = self.transformer(self.add_pos(self.src_emb(src)),
                               self.add_pos(self.tgt_emb(tgt)),
                               tgt_mask=mask, tgt_is_causal=True)
        return self.fc(out)

model   = Seq2SeqTransformer(V)
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

# ── Training with teacher forcing ─────────────────────────────────────────
for epoch in range(20):
    model.train(); el = 0; nb_ = 0
    perm = torch.randperm(len(X_src))
    for i in range(0, len(X_src), 128):
        b = perm[i:i+128]
        logits = model(X_src[b], tgt_in[b])                 # (B, T, V)
        loss   = loss_fn(logits.reshape(-1, V), tgt_out[b].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step(); el += loss.item(); nb_ += 1
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1}: mean batch loss={el/nb_:.4f}')

# ── Greedy decoding: generate the output one character at a time ──────────
def restore(src_text, max_len=None):
    model.eval()
    max_len = max_len or MAX_TGT
    src = torch.tensor([encode_src(src_text)], dtype=torch.long)
    tgt = torch.tensor([[SOS]], dtype=torch.long)          # start with SOS
    for _ in range(max_len):
        with torch.no_grad():
            logits = model(src, tgt)
        nxt = logits[0, -1].argmax().item()                 # most likely next token
        if nxt in (EOS, PAD):
            break                                           # model says it is done
        tgt = torch.cat([tgt, torch.tensor([[nxt]])], dim=1)
    return ''.join(i2c.get(i, '') for i in tgt[0, 1:].tolist())

# ── Evaluation on 200 UNSEEN real sentences ───────────────────────────────
sample = test_pairs[:200]
preds = [restore(a) for a, _ in sample]
exact = sum(p == b for p, (_, b) in zip(preds, sample))
# per-character accuracy: how much of each sentence is right, not just all-or-nothing
char_hits = sum(sum(x == y for x, y in zip(p, b)) for p, (_, b) in zip(preds, sample))
char_tot  = sum(len(b) for _, b in sample)

print('\nHeld-out predictions (source | model | author):')
for (a, b), p in list(zip(sample, preds))[:6]:
    print(f'  {a!r}\n    model : {p!r}\n    author: {b!r}  {"✓" if p == b else "✗"}')
print(f'\nExact-match accuracy on {len(sample)} unseen real sentences: '
      f'{exact}/{len(sample)} = {exact/len(sample):.1%}')
print(f'Per-character accuracy: {char_hits}/{char_tot} = {char_hits/char_tot:.1%}')
print('\nRead the failures before judging the model: most are a missing comma or')
print('a "." where the author wrote "!" — choices no amount of training can')
print('recover from the stripped input, because the information is not there.')


PyTorch 2.13.0


Real sentence pairs extracted: 2,294
Examples (source → target):
  'how brave theyll all think me at home' → "How brave they'll all think me at home!"
  'would the fall never come to an end' → 'Would the fall NEVER come to an end!'
  'dinah my dear' → 'Dinah my dear!'
  'i wish you were down here with me' → 'I wish you were down here with me!'

Vocabulary: 61 tokens (58 characters + PAD/SOS/EOS)
Longest source: 39 chars | longest target: 40 chars
Train: 2,064 sentences | held-out test: 230
Tensors — src (2064, 39) | tgt_in (2064, 41)


Epoch 5: mean batch loss=1.9837


Epoch 10: mean batch loss=0.4739


Epoch 15: mean batch loss=0.2516


Epoch 20: mean batch loss=0.1803



Held-out predictions (source | model | author):
  'i ought to travel'
    model : 'I ought to travel.'
    author: 'I ought to travel.'  ✓
  'so goodby my dear'
    model : 'So goodby My dear.'
    author: 'So goodby, my dear.'  ✗
  'then came an interruption and a change'
    model : 'Then came an interruption and a change.'
    author: 'Then came an interruption and a change.'  ✓
  'it will be giving him so much pleasure'
    model : 'It will be giving him so much pleasure.'
    author: 'It will be giving him so much pleasure!'  ✗
  'i was miserable'
    model : 'I was miserable.'
    author: 'I was miserable.'  ✓
  'here he certainly saved the situation'
    model : 'Here he certainly saved the situation.'
    author: 'Here he certainly saved the situation.'  ✓

Exact-match accuracy on 200 unseen real sentences: 98/200 = 49.0%
Per-character accuracy: 4472/5479 = 81.6%

Read the failures before judging the model: most are a missing comma or
a "." where the author wrote "!" — choices

## 💬 Discuss

1. The model scored **49.0% exact match** and **81.6% per-character**. Which number goes in a proposal to a subtitling client, and which one will the client actually *feel* when they read the output? Justify putting either in a contract.
2. The model wrote `It will be giving him so much pleasure.` where Austen wrote `...pleasure!`. Is that an error? Write a definition of "correct" for this task, then say whether your definition is measurable at scale.
3. Whisper made this component largely redundant. What would have to be true about a client's pipeline in 2026 for you to still build it? (Consider languages, on-device constraints, and existing recognisers you cannot replace.)


## 🌍 Related Worked Example — Decoder-Only Generation

The system above is an **encoder–decoder**: a real sentence, stripped, goes in; the restored sentence comes out. The other dominant design is **decoder-only** (GPT): no separate input encoder — the "input" is simply the start of the sequence and the model continues it. For contrast, revisit the decoder-only char-level generator in **example 01**, which is trained on real Usenet posts and real 911 dispatch text. The references below cover both designs.

**Where each one fits:** encoder–decoder wins when input and output are different strings that must stay aligned (translation, summarisation, the truecasing task above). Decoder-only wins when the output is simply a continuation of the input (chat, completion).


## ⚠️ Where this breaks

**Part of this task is irreducibly ambiguous.** Nothing in the stripped string `it will be giving him so much pleasure` says whether the author wrote `.` or `!`. Exact-match therefore has a ceiling below 100% that no model and no volume of data removes. Report per-character accuracy beside it, or you are measuring the ambiguity and calling it model quality.

**Greedy decoding cannot repair itself.** One path, no backtracking: a wrong character early is a wrong sentence. Beam search recovers some of that at a compute cost — worth it when the output is read by a person, wasteful when it is consumed by another model.

**Character-level attention is quadratic in sequence length.** This works on 40-character sentences and stops being the right tool on paragraphs. Subword tokenization is the standard fix, and it is why T5 and BART are subword models.

**Check upstream before you build.** If the client's recogniser already emits punctuation (Whisper-class), this component is dead weight. The most valuable thing an engineer does on a task like this is establish that it is still a task.


## 📚 References & Further Reading

**Papers:**
- Vaswani et al. (2017) — [Attention Is All You Need](https://arxiv.org/abs/1706.03762) *(the Transformer; §3.5 explains positional encodings)*
- Raffel et al. (2020) — [T5: Exploring the Limits of Transfer Learning](https://arxiv.org/abs/1910.10683) *(text-to-text framing)*
- Lewis et al. (2020) — [BART](https://arxiv.org/abs/1910.13461)

**Docs:** [`nn.Transformer`](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html)


## 📝 Summary

In **04 — Building a Text-to-Text Generation System** you built an encoder–decoder Transformer with SOS/EOS handling, teacher forcing, a causal target mask, **learned positional embeddings**, and greedy decoding — then evaluated it on **held-out real sentences** from published books, at a real production task (truecasing and punctuation restoration, the last stage of a speech-recognition pipeline).

Two things to take away:
1. **Positional embeddings are load-bearing.** Without order information, attention cannot express "capitalise the first letter" or "punctuate the last" — the task becomes unlearnable.
2. **Read the accuracy honestly.** The model scored **49.0% exact match** but **81.6% per-character** on 200 unseen real sentences. That gap is the finding: it reconstructs most of every sentence and then loses the whole point on one mark. And the misses are largely irreducible — it wrote `It will be giving him so much pleasure.` where Austen wrote `...pleasure!`, and `So goodby My dear.` where Carroll wrote `So goodby, my dear.` Nothing in the stripped input carries that information. A synthetic task engineered to be perfectly solvable (a 100% score on reversing invented digit strings) would have hidden this entire lesson about **irreducible ambiguity in real data**.

T5 and BART are this same architecture at scale, trained on this same "text in → text out" framing.
